# 05 Hybrid Retrieval & Query Transformation Strategies

**Real-World Scenario**: Enterprise Helpdesk Ticket Search combining Keyword Part SKUs and Natural Language Issues.

This notebook implements **Reciprocal Rank Fusion (RRF)** merging Sparse BM25 and Dense FAISS vector search results.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Reciprocal Rank Fusion (RRF) Implementation

In [ ]:
def reciprocal_rank_fusion(bm25_ranks: dict, vector_ranks: dict, k: int = 60) -> dict:
    scores = {}
    all_docs = set(bm25_ranks.keys()).union(set(vector_ranks.keys()))
    for doc in all_docs:
        r_bm25 = bm25_ranks.get(doc, 999)
        r_vec = vector_ranks.get(doc, 999)
        scores[doc] = (1.0 / (k + r_bm25)) + (1.0 / (k + r_vec))
    return dict(sorted(scores.items(), key=lambda x: x[1], reverse=True))

# Ranks retrieved by Sparse BM25 (exact SKU match) and Dense Vector (semantic intent match)
bm25_results = {"Ticket_SKU_9021": 1, "Ticket_General_A": 2, "Ticket_General_B": 5}
vector_results = {"Ticket_General_A": 1, "Ticket_General_B": 2, "Ticket_SKU_9021": 4}

rrf_merged = reciprocal_rank_fusion(bm25_results, vector_results)

print("=== Reciprocal Rank Fusion (RRF) Output Ranks ===")
for rank_idx, (doc, score) in enumerate(rrf_merged.items(), 1):
    print(f"Rank #{rank_idx}: {doc} (RRF Score: {score:.6f})")


## Step 3: Indexing Helpdesk Tickets into Hidden `.vectordb/` Directory

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

tickets = [
    "Ticket SKU-9021: Motherboard replacement required for server rack unit 4.",
    "Ticket General-A: System crashing on startup after OS patch update.",
    "Ticket General-B: Slow network transfer speeds across VPN tunnel."
]

if os.getenv("OPENAI_API_KEY"):
    vectorstore = FAISS.from_texts(tickets, OpenAIEmbeddings())
    save_path = os.path.join(".vectordb", "05_hybrid_index")
    vectorstore.save_local(save_path)
    print(f"=== Hybrid Index Saved to Hidden Folder: {save_path} ===")
